In [ ]:
# Install Dependencies
%pip install anthropic python-dotenv

In [ ]:
# Setup: load credentials, build the client.
import os
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# find_dotenv(usecwd=True) walks up from the notebook, so .env is found whether it
# lives in notebooks/ or the repo root. override=True picks up a rotated key.
load_dotenv(find_dotenv(usecwd=True), override=True)

client = Anthropic()          # reads ANTHROPIC_API_KEY from the environment
model = "claude-sonnet-5"

k = os.environ["ANTHROPIC_API_KEY"]
print(f"key {k[:13]}...{k[-4:]} (len={len(k)})  |  model {model}")


In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


In [ ]:
#Make a request
def chat(messages):
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    # response.content is a LIST OF BLOCKS (thinking, text, tool_use, ...).
    # Never assume content[0] is text - branch on block.type.
    return "".join(b.text for b in response.content if b.type == "text")


In [ ]:
# Make a Starting list of messages
messages = []

# Add in the initial user question
add_user_message(messages, "Define quantum computing in one sentance")

#Pass the list of messages into 'chat' to get an answer
answer = chat(messages)

#Take the answer and add it to assistant message
add_assistant_message(messages, answer)

#Add in user followup question
add_user_message(messages, "Write another sentance")

answer = chat(messages)

answer

In [ ]:
messages = []

while True:
    # Get user input
    user_input = input("> ")

    # Let the user leave the loop
    if user_input.strip().lower() in {"quit", "exit"}:
        break

    # Add user input to the list of messages
    add_user_message(messages, user_input)

    # Call Claude with the 'chat' function
    answer = chat(messages)

    # Add generated text to the list of messages
    add_assistant_message(messages, answer)

    # Print the generated text
    print("---")
    print(answer)
    print("---")